<a href="https://colab.research.google.com/github/Tharul-Fernando/CM2604-Telco-Customer-Churn/blob/main/03_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

# Load Original Dataset
file_path = ("/content/drive/MyDrive/ML_Coursework/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(file_path)

# Keep the original data unchanged
df_clean = df.copy()

print("Dataset shape:", df_clean.shape)
display(df_clean.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Convert TotalCharges from text to numerical values
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# Check the new data type
print("Data types:", df_clean["TotalCharges"].dtypes)

# Count missing values after conversion
print("Missing TotalCharges:", df_clean["TotalCharges"].isnull().sum())

Data types: float64
Missing TotalCharges: 11


In [ ]:
# Examine customers with missing values
missing_rows = df_clean[df_clean["TotalCharges"].isnull()]

print("Number of affected customers:", len(missing_rows))

print("Tenure values:", missing_rows["tenure"].unique())

display(missing_rows[["tenure","MonthlyCharges","TotalCharges","Churn"]])

Number of affected customers: 11
Tenure values: [0]


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [ ]:
# Replace missing TotalCharges values with zero
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

# Verify that no missing values remain
print("Missing TotalCharges:", df_clean["TotalCharges"].isnull().sum())

# Verify the datatype
print("Data type:",df_clean["TotalCharges"].dtypes)

# Check the dataset dimensions
print("Dataset shape:", df_clean.shape)

Missing TotalCharges: 0
Data type: float64
Dataset shape: (7043, 21)


### Preprocessing 1: Handling Missing TotalCharges Values

**Problem:** The TotalCharges column was stored as an object and contained 11 blank values, even though the initial null-value check detected no missing data.

**Action:** The column was converted to a numerical data type using `pd.to_numeric()` with `errors="coerce"`. The resulting 11 missing values `(NaN)` were then replaced with zero using `fillna(0)`.

**Justification:** All 11 affected customers had a tenure of zero months, suggesting they may not have accumulated any charges. Replacing the missing values with zero was considered a reasonable assumption, allowing all 7,043 customer records to be retained.

In [ ]:
# Separate the input features and target variable
X = df_clean.drop(columns = ["customerID","Churn"])
y = df_clean["Churn"]

# Check the dimensions
print("Features (X):", X.shape)
print("Target (y):", y.shape)

# Check the feature names
print("\nFeature names:")
print(X.columns.tolist())

# Check the target classes
print("\nTarget classes:")
print(y.unique())

Features (X): (7043, 19)
Target (y): (7043,)

Feature names:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Target classes:
['No' 'Yes']


In [ ]:

# Convert Churn to binary values
y = y.map({"No": 0, "Yes": 1})

# Verify the conversion
print(y.value_counts())

# Check for missing values
print("\nMissing target values:", y.isnull().sum())


Churn
0    5174
1    1869
Name: count, dtype: int64

Missing target values: 0


### Preprocessing 2: Feature and Target Preparation

**Problem:** The dataset contained a unique customer identifier and a categorical target variable that required preparation before model training.

**Action:** The `customerID` column was excluded from the input features. The `Churn` column was separated as the target variable and encoded using 0 for `No` and 1 for `Yes`.

**Justification:** Customer IDs do not provide meaningful information for predicting churn. Excluding them prevents models from learning arbitrary patterns, while binary encoding provides a consistent numerical representation of the target classes that can be used to calculate evaluation metrics later.

In [ ]:
# Import the function used to split the dataset
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )

# Display the dimension of each set
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [ ]:
# Check the churn distribution in the training set
print("Training set churn distribution:")
print(y_train.value_counts(normalize=True) * 100)

print()

# Check the churn distribution in the testing set
print("Testing set churn distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training set churn distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing set churn distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


### Preprocessing 3: Train/Test Splitting

**Problem:** The dataset needed to be divided into separate training and testing sets to evaluate how well the machine learning models perform on unseen customer records.

**Action:** The dataset was split using `train_test_split()` from Scikit-learn, allocating 80% of the records for training and 20% for testing. A fixed `random_state=42` was used for reproducibility, and `stratify=y` was applied to preserve the churn-class proportions.

**Justification:** Separating training and testing data allows the models to be evaluated on records that were not used during training. Stratification helps maintain a similar distribution of churned and non-churned customers in both sets, supporting a more representative evaluation.

In [ ]:
# Get categorical columns
cat_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# SeniorCitizen is categorical
cat_cols.append("SeniorCitizen")

# Get numerical columns
num_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Remove SeniorCitizen from numerical columns
num_cols.remove("SeniorCitizen")

print("Categorical columns:", cat_cols)
print("\nNumerical columns:", num_cols)


Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']

Numerical columns: ['tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
# Import Encoding Libraries
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
# Create the encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apply encoding to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, cat_cols),
        ("num", "passthrough", num_cols)
    ],
    verbose_feature_names_out=False
)


In [ ]:
# Encode training data
train_enc = preprocessor.fit_transform(X_train)

# Encode testing data
test_enc = preprocessor.transform(X_test)

print("Training:", train_enc.shape)
print("Testing:", test_enc.shape)


Training: (5634, 46)
Testing: (1409, 46)


In [ ]:
# Get encoded column names
cols = preprocessor.get_feature_names_out()

# Create a readable table
preview = pd.DataFrame(
    train_enc,
    columns=cols,
    index=X_train.index
)

# Display first five rows
display(preview.head())

# Check missing values
print("Missing values:", pd.isnull(train_enc).sum())

# Count features
print("Total features:", len(cols))


,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,...,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen_0,SeniorCitizen_1,tenure,MonthlyCharges,TotalCharges
3738,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,35.0,49.20,1701.65
3151,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,15.0,75.10,1151.55
4860,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,13.0,40.55,590.35
3867,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,26.0,73.50,1905.70
3810,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,44.55,44.55


Missing values: 0
Total features: 46


### Preprocessing 4: Categorical Feature Encoding

**Problem:** The dataset contained categorical features such as `Contract`, `InternetService` and `PaymentMethod` that needed to be converted into numerical values before model training.

**Action:** Categorical and numerical features were identified separately. `SeniorCitizen` was treated as categorical because it represents a binary category. One-hot encoding was applied using `OneHotEncoder` and `ColumnTransformer`, while the numerical features were left unchanged.

**Justification:** One-hot encoding converts categories into numerical representations without assigning an artificial ordering. The encoder was fitted using only the training data, and the same transformation was applied to the testing data to prevent test-data leakage.

In [ ]:
# Import the feature scaling tool
from sklearn.preprocessing import StandardScaler

In [ ]:
# Convert encoded arrays into DataFrames
train_scaled = pd.DataFrame(
    train_enc,
    columns = cols,
    index = X_train.index
)

test_scaled = pd.DataFrame(
    test_enc,
    columns = cols,
    index = X_test.index
)

print("Training:", train_scaled.shape)
print("Testing:", test_scaled.shape)

Training: (5634, 46)
Testing: (1409, 46)


In [ ]:
# Create the scaler
scaler = StandardScaler()

# Scale numerical columns in training data
train_scaled[num_cols] = scaler.fit_transform(
    train_scaled[num_cols]
)

# Scale numerical columns in testing data
test_scaled[num_cols] = scaler.transform(
    test_scaled[num_cols]
)

print("Feature scaling completed!")

Feature scaling completed!


In [ ]:
# Compare numerical features before and after scaling
print("Before scaling:")
display(
    pd.DataFrame(
        train_enc,
        columns=cols
    )[num_cols].head()
)

print("After scaling:")
display(train_scaled[num_cols].head())

# Check the mean and standard deviation
print("\nMean after scaling:")
print(train_scaled[num_cols].mean().round(2))

print("\nStandard deviation after scaling:")
print(train_scaled[num_cols].std(ddof=0).round(2))


Before scaling:


,tenure,MonthlyCharges,TotalCharges
0,35.0,49.20,1701.65
1,15.0,75.10,1151.55
2,13.0,40.55,590.35
3,26.0,73.50,1905.70
4,1.0,44.55,44.55


After scaling:


,tenure,MonthlyCharges,TotalCharges
3738,0.102371,-0.521976,-0.262257
3151,-0.711743,0.337478,-0.503635
4860,-0.793155,-0.809013,-0.749883
3867,-0.263980,0.284384,-0.172722
3810,-1.281624,-0.676279,-0.989374



Mean after scaling:
tenure           -0.0
MonthlyCharges    0.0
TotalCharges      0.0
dtype: float64

Standard deviation after scaling:
tenure            1.0
MonthlyCharges    1.0
TotalCharges      1.0
dtype: float64


In [ ]:
# Check that no missing values remain
print("Training missing values:",
      train_scaled.isnull().sum().sum())

print("Testing missing values:",
      test_scaled.isnull().sum().sum())

# Check final dataset dimensions
print("Training shape:", train_scaled.shape)
print("Testing shape:", test_scaled.shape)

Training missing values: 0
Testing missing values: 0
Training shape: (5634, 46)
Testing shape: (1409, 46)


### Preprocessing 5: Feature Scaling

**Problem:** Numerical features such as `tenure`, `MonthlyCharges`, and `TotalCharges` had different value ranges, which could affect algorithms that are sensitive to feature scales.

**Action:** StandardScaler was used to standardize the three numerical features. The scaler was fitted using the training data, and the same transformation was applied to the testing data. One-hot encoded categorical features were left unchanged.

**Justification:** Standardization places numerical features on comparable scales, reducing the influence of differences in magnitude. Fitting the scaler only on the training data also prevents information from the testing set from influencing the preprocessing process.

In [ ]:
# Create a preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Define how the features will be prepared
prep_model = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(handle_unknown = "ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)


In [ ]:
# Create the KNN model with 5 neighbors
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("prep", prep_model),
    ("model", KNeighborsClassifier(n_neighbors = 5))
])

In [ ]:
# Train the KNN model
knn.fit(X_train, y_train)

print("KNN model trained successfully!")

KNN model trained successfully!


In [ ]:
# Evaluate KNN using the five stratified folds
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

knn_scores = cross_val_score(
    knn,
    X_train,
    y_train,
    cv = cv,
    scoring = "f1"
)

print("KNN F1 scores:", knn_scores)
print("Average KNN F1:", round(knn_scores.mean(), 4))

KNN F1 scores: [0.5395189  0.53097345 0.56802721 0.53989813 0.55536627]
Average KNN F1: 0.5468


### Model 1: K-Nearest Neighbors (KNN)

**Method:** KNN was used as the first supervised classification algorithm to predict customer churn.

**Implementation:** A Scikit-learn pipeline was created to perform categorical encoding, numerical scaling and KNN classification. The initial model used five nearest neighbors (`n_neighbors=5`).

**Justification:** KNN classifies customers based on their similarity to existing labelled customers. Feature scaling was applied because KNN uses distance calculations and can be affected by differences in numerical feature ranges.

**Evaluation:** The initial model will be compared with Logistic Regression and Random Forest. Hyperparameter tuning will be performed using cross-validation on the training dataset.

In [ ]:
# Create the Logistic Regression Model
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

log_reg = Pipeline([
    ("prep", clone(prep_model)),
    ("model", LogisticRegression(max_iter = 1000))
])

In [ ]:
# Train the Logistic Regression model
log_reg.fit(X_train, y_train)
print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [ ]:

# Evaluate Logistic Regression
log_scores = cross_val_score(
    log_reg,
    X_train,
    y_train,
    cv = cv,
    scoring = "f1"
)

print("F1 scores:", log_scores)
print("Average F1:", round(log_scores.mean(), 4))


F1 scores: [0.58500914 0.54945055 0.58888889 0.64248705 0.59622642]
Average F1: 0.5924


### Model 2: Logistic Regression

**Method:** Logistic Regression was implemented as the second supervised classification algorithm to predict customer churn.

**Implementation:** A Scikit-learn pipeline was used to combine categorical encoding, numerical scaling and Logistic Regression. The model used `max_iter=1000` to allow sufficient iterations for optimization.

**Justification:** Logistic Regression is suitable for binary classification because it estimates the probability that a customer belongs to the churn class. F1-score was selected for the initial cross-validation evaluation because the dataset contains an imbalanced target distribution.

**Evaluation:** Five-fold stratified cross-validation was performed using the training data, with F1-score as the evaluation metric. This provides an initial baseline for comparing Logistic Regression with KNN and Random Forest before hyperparameter tuning and final testing.

In [ ]:
# Create the Random Forest model
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", clone(prep_model)),
    ("model", RandomForestClassifier(
        n_estimators = 100,
        random_state = 42
        ))
])


In [ ]:
# Train the Random Forest model
rf.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [ ]:
# Evaluate Random Forest using cross-validation
rf_scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv = cv,
    scoring = "f1"
)

print("F1 scores:", rf_scores)
print("Average F1:", round(rf_scores.mean(), 4))

F1 scores: [0.52059925 0.5199241  0.58052434 0.53483992 0.5518591 ]
Average F1: 0.5415


In [ ]:
# Compare baseline cross-validation results
print("Logistic Regression F1:", round(log_scores.mean(), 4))
print("Random Forest F1:", round(rf_scores.mean(), 4))

Logistic Regression F1: 0.5924
Random Forest F1: 0.5415


### Model 3: Random Forest

**Method:** Random Forest was implemented as the third supervised classification algorithm to predict customer churn.

**Implementation:** A Scikit-learn pipeline was created to combine data preprocessing and Random Forest classification. The baseline model used 100 decision trees (`n_estimators=100`) and `random_state=42` for reproducibility.

**Justification:** Random Forest combines predictions from multiple decision trees, which can improve generalization and reduce overfitting compared with an individual decision tree.

**Evaluation:** Five-fold stratified cross-validation was performed using the training dataset, with F1-score as the evaluation metric. The results will be compared with KNN and Logistic Regression before hyperparameter tuning and final testing.

In [ ]:
# Compare the three baseline models
baseline_results = pd.DataFrame({
    "Model": ["KNN", "Logistic Regression", "Random Forest"],
    "Average F1": [
        knn_scores.mean(),
        log_scores.mean(),
        rf_scores.mean()
    ]
})

# Sort models by F1-score
baseline_results = baseline_results.sort_values(
    by="Average F1",
    ascending=False
)

display(baseline_results.round(4))

,Model,Average F1
1,Logistic Regression,0.5924
0,KNN,0.5468
2,Random Forest,0.5415


### Baseline Model Comparison

The three classification models were evaluated using five-fold stratified cross-validation with F1-score as the primary performance metric.

Logistic Regression achieved the highest average F1-score of 0.5924, followed by KNN with 0.5468 and Random Forest with 0.5415.

These results indicate that Logistic Regression performs best among the initial configurations. However, hyperparameter tuning will be performed to investigate whether the predictive performance of each model can be improved.

## Hyperparameter Tuning

Hyperparameter tuning was performed to identify better configurations for KNN, Logistic Regression and Random Forest. GridSearchCV was used with five-fold stratified cross-validation and F1-score as the primary evaluation metric.

The search was conducted using only the training dataset to avoid using test data for model selection.

### KNN Hyperparameter Tuning

The number of neighbors and weighting method were tuned to identify a suitable configuration for customer churn prediction.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter values to test
knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"]
}

# Search for the best KNN settings
knn_search = GridSearchCV(
    estimator = knn,
    param_grid = knn_params,
    cv = cv,
    scoring = "f1",
    n_jobs = -1
)

# Run tunning on training data
knn_search.fit(X_train, y_train)

print("Best KNN parameters:", knn_search.best_params_)
print("Best KNN F1-score:", round(knn_search.best_score_, 4))

Best KNN parameters: {'model__n_neighbors': 15, 'model__weights': 'uniform'}
Best KNN F1-score: 0.5847


In [ ]:
# Compare KNN before and after tuning
print("KNN baseline F1:", round(knn_scores.mean(), 4))
print("KNN tuned F1:", round(knn_search.best_score_, 4))

KNN baseline F1: 0.5468
KNN tuned F1: 0.5847


**KNN Tuning Results**

The baseline KNN model achieved an average cross-validation F1-score of 0.5468. After hyperparameter tuning, the best configuration used 15 neighbours (`n_neighbors=15`) with uniform weighting (`weights='uniform'`), achieving an F1-score of 0.5847.

This represents an improvement of 0.0379 in the F1-score. The results indicate that using more neighbours improved the model's performance within the tested parameter range. The held-out test dataset was not used during tuning.

### Logistic Regression Hyperparameter Tuning

Logistic Regression was tuned by testing different values of the regularization parameter (`C`). This parameter controls the strength of regularization applied to the model.

`GridSearchCV` was used with five-fold stratified cross-validation to identify the value that produced the highest average F1-score. Only the training dataset was used during tuning to avoid influencing model selection with the held-out test data.

In [ ]:
# Hyperparameter values to test
log_params = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# Search for the best Logistics Regression settings
log_search = GridSearchCV(
    estimator = log_reg,
    param_grid = log_params,
    cv = cv,
    scoring = "f1",
    n_jobs = -1
)

# Run tuning on training data
log_search.fit(X_train, y_train)

print("Best Logistic Regression parameters:", log_search.best_params_)
print("Best Logistic Regression F1-score:", round(log_search.best_score_, 4))


Best Logistic Regression parameters: {'model__C': 100}
Best Logistic Regression F1-score: 0.5974


In [ ]:
# Compare Logistic Regression before and after tuning
print("Logistic Regression baseline F1:", round(log_scores.mean(), 4))
print("Logistic Regression tuned F1:", round(log_search.best_score_, 4))

Logistic Regression baseline F1: 0.5924
Logistic Regression tuned F1: 0.5974


**Logistic Regression Tuning Results**

The baseline Logistic Regression model achieved an average cross-validation F1-score of 0.5924. After tuning the regularization parameter (`C`), the best value was 100, producing an F1-score of 0.5974.

This represents a small improvement of 0.0050. The results suggest that weaker regularization improved performance within the tested range. The held-out test dataset was not used during tuning.

### Random Forest Hyperparameter Tuning

Random Forest was tuned by testing different values for the number of decision trees (`n_estimators`), maximum tree depth (`max_depth`) and minimum samples required to split a node (`min_samples_split`).

`GridSearchCV` was used with five-fold stratified cross-validation and F1-score as the evaluation metric. Tuning was performed using only the training dataset.

In [ ]:
# Hyperparameter values to test
rf_params = {
    "model__n_estimators": [100,200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5]
}

# Search for the best Random Forest settings
rf_search = GridSearchCV(
    estimator = rf,
    param_grid = rf_params,
    cv = cv,
    scoring = "f1",
    n_jobs = -1
)

# Run tuning on training data
rf_search.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_search.best_params_)
print("Best Random Forest F1-score:", round(rf_search.best_score_, 4))

Best Random Forest parameters: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best Random Forest F1-score: 0.5865


In [ ]:
# Compare Random Forest before and after tuning
print("Random Forest baseline F1:", round(rf_scores.mean(), 4))
print("Random Forest tuned F1:", round(rf_search.best_score_, 4))


Random Forest baseline F1: 0.5415
Random Forest tuned F1: 0.5865


**Random Forest Tuning Results**

The baseline Random Forest model achieved an average cross-validation F1-score of 0.5415. After hyperparameter tuning, the best configuration used 200 decision trees (`n_estimators=200`), a maximum tree depth of 10 (`max_depth=10`) and a minimum of 5 samples required to split a node (`min_samples_split=5`).

The tuned model achieved an average F1-score of 0.5865, representing an improvement of 0.0450. The results indicate that adjusting the tree count and controlling tree complexity improved the model's performance within the tested parameter range. The held-out test dataset was not used during tuning.